In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "0114bae219d4d94bc9d18a2e1ab76c089e2dc890"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/0114bae219d4d94bc9d18a2e1ab76c089e2dc890/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Pooling

Many image tasks require a global prediction, such as whether an image
contains a cat. The final representation must therefore combine information
from the entire input. Convolutional networks commonly build this
representation by progressively reducing spatial resolution while increasing
the receptive field of each hidden unit.
Deeper hidden units have larger receptive fields relative to the input.
Reducing spatial resolution accelerates this growth because subsequent kernels
cover a larger effective area.

For lower-level features such as edges, we also want small translations of the
input to produce modest changes in the representation.
For instance, if we take the image `X`
with a sharp delineation between black and white
and shift the whole image by one pixel to the right,
i.e., `Z[i, j] = X[i, j - 1]`,
then the edge in the new output shifts by one pixel and sampled values may
change.
Such shifts occur whenever an object or camera moves slightly.

This section introduces *pooling layers*, which summarize local windows to
reduce spatial resolution and sensitivity to small translations.

In [ ]:
from d2l import tensorflow as d2l
import tensorflow as tf

## Maximum Pooling and Average Pooling

Like convolutional layers, *pooling* operators
consist of a fixed-shape window that is slid over
all regions in the input according to its stride,
computing a single output for each location traversed
by the fixed-shape window (sometimes known as the *pooling window*).
However, unlike the cross-correlation computation
of the inputs and kernels in the convolutional layer,
the pooling layer contains no parameters (there is no *kernel*).
Instead, pooling operators are deterministic,
typically calculating either the maximum or the average value
of the elements in the pooling window.
These operations are called *maximum pooling* (*max-pooling* for short)
and *average pooling*, respectively.

*Average pooling* dates to the earliest CNNs and resembles image downsampling.
Instead of retaining every second or third pixel, it averages adjacent pixels,
combining their information and reducing high-frequency variation.
*Max-pooling* was introduced in
@Riesenhuber.Poggio.1999 in the context of cognitive neuroscience to describe 
how information might be aggregated hierarchically for the purpose 
of object recognition; there already was an earlier version in speech recognition
[@Yamaguchi.Sakamoto.Akabane.ea.1990]. Historically, max-pooling was
often preferred inside convolutional bodies, while average pooling became the
standard choice for global classifier heads. Neither dominates in every setting.

In both cases, as with the cross-correlation operator,
we can think of the pooling window
as starting from the upper-left of the input tensor
and sliding across it from left to right and top to bottom.
At each location that the pooling window hits,
it computes the maximum or average
value of the input subtensor in the window,
depending on whether max or average pooling is employed.


![Max-pooling with a pooling window shape of $2\times 2$. The shaded portions are the first output element as well as the input tensor elements used for the output computation: $\max(0, 1, 3, 4)=4$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/pooling.svg)

The output tensor in the figure  has a height of 2 and a width of 2.
The four elements are derived from the maximum value in each pooling window:

$$
\begin{aligned}
\max(0, 1, 3, 4) &= 4,\\
\max(1, 2, 4, 5) &= 5,\\
\max(3, 4, 6, 7) &= 7,\\
\max(4, 5, 7, 8) &= 8.
\end{aligned}
$$

More generally, we can define a $p \times q$ pooling layer by aggregating over 
a region of that size. Returning to the problem of edge detection, 
we use the output of the convolutional layer
as input for $2\times 2$ max-pooling.
Denote the edge-detector output by `X` and the pooling output by `Y`. If a
single response of value 1 moves anywhere within the same $2\times2$ pooling
window while the other responses remain no larger, then `Y[i, j]` remains 1.
Thus max-pooling can make a representation insensitive to some small movements
that stay within a window. Crossing a window boundary can still change the
pooled output abruptly, so this is local tolerance rather than exact translation
invariance.

In the code below, we implement the forward propagation
of the pooling layer in the `pool2d` function.
This function is similar to the `corr2d` function
in that section.
No kernel is needed: the output is
the maximum or the average of each region in the input.

In [ ]:
def pool2d(X, pool_size, mode='max'):
    p_h, p_w = pool_size
    Y = tf.Variable(tf.zeros((X.shape[0] - p_h + 1, X.shape[1] - p_w +1)))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            if mode == 'max':
                Y[i, j].assign(tf.reduce_max(X[i: i + p_h, j: j + p_w]))
            elif mode =='avg':
                Y[i, j].assign(tf.reduce_mean(X[i: i + p_h, j: j + p_w]))
    return Y

We can construct the input tensor `X` in the figure to validate the output of the two-dimensional max-pooling layer.

In [ ]:
X = d2l.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
pool2d(X, (2, 2))

We can likewise evaluate average pooling.

In [ ]:
pool2d(X, (2, 2), 'avg')

## Padding and Stride

As with convolutional layers, pooling layers
change the output shape.
And as before, we can adjust the operation to achieve a desired output shape
by padding the input and adjusting the stride.
We can demonstrate the use of padding and strides
in pooling layers via the built-in two-dimensional max-pooling layer from the deep learning framework.
We first construct an input tensor `X` whose shape has four dimensions,
where the number of examples (batch size) and number of channels are both 1.

Unlike the other frameworks, TensorFlow
prefers and is optimized for *channels-last* input.

In [ ]:
X = d2l.reshape(d2l.arange(16, dtype=d2l.float32), (1, 4, 4, 1))
X

Since pooling aggregates information from an area, deep learning frameworks default to matching pooling window sizes and stride. For instance, if we use a pooling window of shape `(3, 3)`
we get a stride shape of `(3, 3)` by default.

In [ ]:
pool2d = tf.keras.layers.MaxPool2D(pool_size=[3, 3])
# Pooling has no model parameters, hence it needs no initialization
pool2d(X)

The stride and padding can be specified explicitly to override the framework
defaults.

In [ ]:
paddings = tf.constant([[0, 0], [1,0], [1,0], [0,0]])
X_padded = tf.pad(X, paddings, "CONSTANT")
pool2d = tf.keras.layers.MaxPool2D(pool_size=[3, 3], padding='valid',
                                   strides=2)
pool2d(X_padded)

Pooling windows can also have different heights and widths.

In [ ]:
paddings = tf.constant([[0, 0], [0, 0], [1, 1], [0, 0]])
X_padded = tf.pad(X, paddings, "CONSTANT")

pool2d = tf.keras.layers.MaxPool2D(pool_size=[2, 3], padding='valid',
                                   strides=(2, 3))
pool2d(X_padded)

## Multiple Channels

When processing multi-channel input data,
the pooling layer pools each input channel separately,
rather than summing the inputs up over channels
as in a convolutional layer.
This means that the number of output channels for the pooling layer
is the same as the number of input channels.
Below, we will concatenate tensors `X` and `X + 1`
on the channel dimension to construct an input with two channels.

TensorFlow's channels-last layout requires concatenation along the final
dimension.

In [ ]:
# Concatenate along `dim=3` due to channels-last syntax
X = d2l.concat([X, X + 1], 3)
X

The output still has two channels after pooling.

In [ ]:
paddings = tf.constant([[0, 0], [1,0], [1,0], [0,0]])
X_padded = tf.pad(X, paddings, "CONSTANT")
pool2d = tf.keras.layers.MaxPool2D(pool_size=[3, 3], padding='valid',
                                   strides=2)
pool2d(X_padded)

TensorFlow stores these tensors in NHWC order, whereas the MXNet and PyTorch
tabs use NCHW order. Transposing the TensorFlow result from NHWC to NCHW gives
the same values; only the channel axis is displayed in a different position.

## Summary

Pooling is a simple operation: it replaces each window of values with a single
summary, the maximum or the mean. The stride and padding semantics of
convolutions carry over unchanged, and pooling applies to each channel
separately, so the number of channels is preserved. There are no parameters to
learn. A $2 \times 2$ window with stride 2, which quarters the number of
spatial locations, is the classical local configuration; global average
pooling is now the common classification head.

Modern convolutional networks often reduce resolution with *strided
convolutions*: a convolution with stride 2 halves the resolution while learning
its aggregation weights instead of fixing them to a maximum or mean. ResNet
(that section) uses this pattern within its stages. Pooling remains
common in two roles. *Global average pooling*, introduced with the
network-in-network architecture (that section)
[@Lin.Chen.Yan.2013], averages each channel over all spatial positions and
turns the final feature map into one value per channel. It often replaces the
large fully connected classifier stacks of early CNNs. Max-pooling persists in
some network stems and in detection models that combine feature maps across
scales.

One caveat applies to every stride-2 downsampler, pooled or convolutional:
subsampling a signal without first removing its high spatial frequencies can
*alias*, so that shifting the input by a single pixel changes the output
noticeably; applying a small low-pass (blur) filter before subsampling can make
the result substantially more shift-consistent, though not exactly invariant
[@zhang2019making]. Beyond these mainstays there are randomized variants
such as stochastic pooling [@Zeiler.Fergus.2013] and fractional
max-pooling [@Graham.2014], and the attention mechanisms of
that section aggregate by learned alignment
between a query and representation vectors rather than by spatial location.


## Exercises

1. Implement average pooling through a convolution. 
1. Prove that max-pooling cannot be implemented through a convolution alone. 
1. Max-pooling can be accomplished using ReLU operations, i.e., $\textrm{ReLU}(x) = \max(0, x)$.
    1. Express $\max (a, b)$ by using only ReLU operations.
    1. Use this to implement max-pooling by means of convolutions and ReLU layers. 
    1. How many channels and layers do you need for a $2 \times 2$ convolution? How many for a $3 \times 3$ convolution?
1. What is the computational cost of the pooling layer? Assume that the input to the pooling layer is of size $c\times h\times w$, the pooling window has a shape of $p_\textrm{h}\times p_\textrm{w}$ with a padding of $(p_\textrm{h}, p_\textrm{w})$ and a stride of $(s_\textrm{h}, s_\textrm{w})$.
1. Why do you expect max-pooling and average pooling to work differently?
1. Do we need a separate minimum pooling layer? Can you replace it with another operation?
1. We could use the softmax operation for pooling. Why might it not be so popular?
1. Naive stride-2 downsampling keeps every second entry of its input. Apply it to the one-dimensional signals $(1, 0, 1, 0, 1, 0)$ and $(0, 1, 0, 1, 0, 1)$, i.e., the same pattern shifted by one position. 
    1. Compare the two outputs. Why is this behavior called *aliasing*, and which input frequencies can a stride-2 subsampler represent faithfully?
    1. Blur-pool counteracts aliasing by applying a small averaging filter before subsampling. Work out what blur-pool with a two-tap box filter $(\tfrac{1}{2}, \tfrac{1}{2})$ computes on the two signals above. Which pooling operation from this section does it coincide with?

[Discussions](https://d2l.discourse.group/t/274)